# Aula 3: fine-tuning leve com QLoRA e Unsloth (Demo 5)

**Grandes Modelos de Linguagem · Residência em Engenharia de Software Inteligente (FACOM/UFMS)**\
Prof. Bruno Nogueira · Aula 3 (síncrona) · 18/11/2026

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmnogueira-ufms/LLM-ESIA-2026/blob/main/aula03/aula03_unsloth_lora.ipynb)

Vamos ensinar a **triagem da Secretaria Digital** a um modelo pequeno e aberto (Qwen2.5 com 1,5 B de parâmetros), com **QLoRA**: o modelo base fica congelado e guardado em **4 bits**, e treinamos só os adaptadores **LoRA**. Usamos a biblioteca **Unsloth**, que reduz a memória e o tempo de treino e roda numa GPU T4 do Colab.

| Passo | O que acontece |
|---|---|
| 1 | dados: 40 chamados de treino da Aula 2 + 100 novos; teste: os **mesmos 20** das Aulas 2 e 3 |
| 2 | carregar o modelo em 4 bits e medir a triagem **antes** do ajuste |
| 3 | acrescentar o LoRA (r = 16) e treinar só na resposta |
| 4 | medir **depois**, com o mesmo prompt e o mesmo teste; ver a Ana, a Júlia e o Carlos |
| 5 | salvar o adaptador (dezenas de MB) e, opcionalmente, exportar para o Ollama |

> **Antes de começar:** *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. O treino leva alguns minutos: na aula, rodamos tudo antes do intervalo.

> **Sobre os dados:** 140 exemplos é pouco para fine-tuning. O objetivo aqui é ver a **mecânica** (e o efeito no formato da saída), não bater recorde. Em um caso real, pense em centenas a milhares de exemplos bem rotulados.

## 0. Instalação

In [ ]:
# No Colab, basta instalar o unsloth (ele traz transformers, trl, peft e bitsandbytes compatíveis).
# Se aparecer erro de versão depois da instalação: Ambiente de execução → Reiniciar sessão, e rode de novo a partir daqui.
!pip install -q unsloth

In [ ]:
import time, re, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

REPO = "https://raw.githubusercontent.com/bmnogueira-ufms/LLM-ESIA-2026/main"
arquivo = lambda c: c if Path(c).exists() else f"{REPO}/{c}"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (ative a T4)")

---
## 1. Dados

Mesma divisão da Aula 2 (`random_state=42`): o teste continua sendo os **mesmos 20 chamados**, então os números são comparáveis com a Demo 2. O treino junta os 40 chamados de treino da Aula 2 com 100 chamados novos (`dados/chamados_treino_extra.csv`), 25 por categoria.

In [ ]:
df = pd.read_csv(arquivo("dados/chamados_exemplo.csv"))
treino_aula2, teste = train_test_split(df, test_size=20, stratify=df["categoria"], random_state=42)
extra = pd.read_csv(arquivo("dados/chamados_treino_extra.csv"))
treino = pd.concat([treino_aula2, extra], ignore_index=True).sample(frac=1, random_state=0)

assert not set(treino["texto"]) & set(teste["texto"]), "chamado do teste no treino!"
CATEGORIAS = sorted(df["categoria"].unique())
print(f"treino: {len(treino)} chamados · teste: {len(teste)} chamados")
print(treino["categoria"].value_counts().to_dict())

ANA = ("Oi! Paguei o boleto da rematrícula na segunda e no portal ainda aparece pendente. "
       "Preciso resolver até sexta, senão perco a vaga em Cálculo II. Meu RGA é [RGA].")     # RGA já mascarado
JULIA = "Errei a senha várias vezes, travou tudo e agora não consigo entrar no AVA. Tenho prova amanhã!"
CARLOS = "Quero trancar o semestre. Até quando dá e quantas vezes posso trancar?"
MENSAGENS = {"Ana": ANA, "Júlia": JULIA, "Carlos": CARLOS}

O prompt é **curto de propósito**: só as categorias e o formato, sem descrições nem regra de fronteira. Queremos ver o que o modelo aprende **com os exemplos**, e não com as instruções.

In [ ]:
SISTEMA = ("Você faz a triagem de mensagens de estudantes da secretaria acadêmica. "
           "Categorias: " + ", ".join(CATEGORIAS) + ". "
           'Responda somente com um JSON no formato {"categoria": "<categoria>"}.')

def mensagens(texto, resposta=None):
    msgs = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": texto}]
    if resposta is not None:
        msgs.append({"role": "assistant", "content": json.dumps({"categoria": resposta})})
    return msgs

print(json.dumps(mensagens(treino["texto"].iloc[0], treino["categoria"].iloc[0]), ensure_ascii=False, indent=1))

---
## 2. O modelo em 4 bits e a triagem **antes** do ajuste

`unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit` já vem quantizado em 4 bits (NF4). Repare na memória ocupada: o modelo em 16 bits ocuparia cerca de 3 GB.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=512,      # nossos exemplos têm menos de 150 tokens
    load_in_4bit=True,       # QLoRA: base em 4 bits
)
print(f"memória de GPU ocupada: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
def classificar(texto, max_new_tokens=20):
    ids = tokenizer.apply_chat_template(mensagens(texto), add_generation_prompt=True, return_tensors="pt").to("cuda")
    inicio = time.time()
    with torch.no_grad():
        out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    texto_saida = tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    return texto_saida, time.time() - inicio

def categoria_do_json(resposta):
    try:
        c = json.loads(resposta[resposta.find("{"): resposta.rfind("}") + 1]).get("categoria")
        return c if c in CATEGORIAS else "invalida"
    except Exception:
        return "invalida"

def avaliar(nome):
    linhas = []
    for texto, rotulo in zip(teste["texto"], teste["categoria"]):
        bruta, lat = classificar(texto)
        linhas.append({"texto": texto, "esperado": rotulo, "bruta": bruta, "previsto": categoria_do_json(bruta), "latencia_s": lat})
    res = pd.DataFrame(linhas)
    tres = {quem: categoria_do_json(classificar(m)[0]) for quem, m in MENSAGENS.items()}
    resumo = {"modelo": nome, "acurácia": (res["previsto"] == res["esperado"]).mean(),
              "JSON válido": (res["previsto"] != "invalida").mean(), "latência média (s)": res["latencia_s"].mean()}
    resumo.update({f"→ {q}": c for q, c in tres.items()})
    return res, resumo

FastLanguageModel.for_inference(model)
res_antes, resumo_antes = avaliar("Qwen 1,5 B em 4 bits, zero-shot")
print(resumo_antes)
res_antes[res_antes["previsto"] != res_antes["esperado"]][["texto", "esperado", "bruta"]]

---
## 3. LoRA e treino

### 3.1 As matrizes e os adaptadores

**Onde estão os pesos.** Cada bloco do Transformer faz quase todo o trabalho com camadas lineares, $h = W x$. No Qwen2.5 1,5 B há 28 blocos, cada um com 7 matrizes: na atenção, `q_proj` ($W_Q$, 1536 × 1536), `k_proj` e `v_proj` ($W_K$ e $W_V$, 256 × 1536) e `o_proj` ($W_O$, 1536 × 1536); na FFN, `gate_proj` e `up_proj` (8960 × 1536) e `down_proj` (1536 × 8960). Ajustar o modelo é mudar os números dessas matrizes.

**O que o LoRA faz.** Em vez de mudar $W_0$, somamos uma correção $\Delta W = B A$:

$$h = W_0 x + \frac{\alpha}{r} B (A x)$$

- $W_0$ é a matriz original: **congelada** (e, no QLoRA, guardada em 4 bits);
- $A$ ($r \times d_{in}$) projeta $x$ em $r$ números: cada linha de $A$ é uma direção que o treino aprende a observar em $x$;
- $B$ ($d_{out} \times r$) transforma esses $r$ números numa correção para as $d_{out}$ saídas: cada coluna de $B$ é um padrão de correção;
- $r$ é o **posto**: quantas direções de mudança independentes $\Delta W$ consegue expressar. Com $r = 16$ e $W_Q$ de 1536 × 1536, $A$ e $B$ somam $2 \times 1536 \times 16 \approx 49$ mil números, contra 2,36 milhões de $W_Q$;
- $A$ começa aleatória e $B$ começa em **zero**: no início, $BA = 0$ e o modelo é idêntico ao original.

`get_peft_model` acrescenta um par $A$, $B$ em cada uma das 196 matrizes. Com `r = 16`, treinamos cerca de 1,2% dos parâmetros; o resto fica congelado em 4 bits.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                                   # posto do adaptador
    lora_alpha=16,                          # escala alpha/r = 1
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",   # economiza memória das ativações
    random_state=3407,
)
FastLanguageModel.for_training(model)
model.print_trainable_parameters()

### 3.2 Dados no formato de chat

Cada exemplo vira o texto completo da conversa (sistema, mensagem do estudante e a resposta em JSON), no template de chat do próprio Qwen.

In [ ]:
from datasets import Dataset

textos = [tokenizer.apply_chat_template(mensagens(t, c), tokenize=False) for t, c in zip(treino["texto"], treino["categoria"])]
ds = Dataset.from_dict({"text": textos})
print(textos[0])
print("tokens no maior exemplo:", max(len(tokenizer(t)["input_ids"]) for t in textos))

### 3.3 Treino (só na resposta)

Usamos o `SFTTrainer` da biblioteca TRL. Com `train_on_responses_only`, a perda conta **só os tokens da resposta** do assistente: o modelo não gasta capacidade aprendendo a repetir o prompt.

| Hiperparâmetro | Valor | Por quê |
|---|---|---|
| épocas | 3 | poucos dados: mais épocas tende a decorar |
| batch efetivo | 8 × 2 = 16 | estabilidade |
| taxa de aprendizado | 2e-4 | típica de LoRA |
| otimizador | adamw_8bit | estados do Adam em 8 bits: menos memória |

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="saida_treino",
        report_to="none",
    ),
)

try:
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                      response_part="<|im_start|>assistant\n")
    print("perda calculada só na resposta do assistente")
except Exception as e:
    print("train_on_responses_only indisponível nesta versão; treinando no texto inteiro:", type(e).__name__)

In [ ]:
inicio = time.time()
estatisticas = trainer.train()
print(f"tempo de treino: {(time.time() - inicio) / 60:.1f} min · memória máxima: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

In [ ]:
import matplotlib.pyplot as plt
log = pd.DataFrame([h for h in trainer.state.log_history if "loss" in h])
plt.figure(figsize=(6, 3))
plt.plot(log["step"], log["loss"], marker="o", ms=3)
plt.xlabel("passo"); plt.ylabel("perda (treino)"); plt.title("Curva de perda")
plt.grid(alpha=0.3); plt.show()

---
## 4. A triagem **depois** do ajuste

Mesmo prompt curto, mesmo teste, mesma forma de contar. Compare com o "antes" e com a Demo 2 (prompt v2 com regra de fronteira).

In [ ]:
FastLanguageModel.for_inference(model)
res_depois, resumo_depois = avaliar("o mesmo + LoRA (QLoRA)")
pd.DataFrame([resumo_antes, resumo_depois]).style.format({"acurácia": "{:.0%}", "JSON válido": "{:.0%}", "latência média (s)": "{:.2f}"})

In [ ]:
# Onde ainda erra?
res_depois[res_depois["previsto"] != res_depois["esperado"]][["texto", "esperado", "previsto", "bruta"]]

**Para observar:**
- O ganho costuma aparecer primeiro no **formato** (JSON sempre válido) e depois no acerto.
- A Ana: os dados de treino têm vários "paguei... e aparece pendente" rotulados como `financeiro`. O modelo aprendeu a regra de fronteira **pelos exemplos**, sem ninguém escrevê-la no prompt?
- Com 20 exemplos de teste, cada acerto vale 5 pontos percentuais.

---
## 5. Salvar e usar o adaptador

In [ ]:
model.save_pretrained("lora_triagem")       # só os adaptadores LoRA
tokenizer.save_pretrained("lora_triagem")
tamanho = sum(f.stat().st_size for f in Path("lora_triagem").rglob("*") if f.is_file()) / 1e6
print(f"pasta lora_triagem: {tamanho:.0f} MB (o modelo base continua o mesmo e não é copiado)")

Para usar depois, carregue o **mesmo modelo base** e aponte para a pasta do adaptador:

```python
model, tokenizer = FastLanguageModel.from_pretrained("lora_triagem", max_seq_length=512, load_in_4bit=True)
FastLanguageModel.for_inference(model)
```

**Exportar para o Ollama (opcional, demora alguns minutos):** o Unsloth junta base + adaptador e grava um arquivo GGUF quantizado, que roda no Ollama ou no llama.cpp, dentro da universidade.

```python
model.save_pretrained_gguf("triagem_gguf", tokenizer, quantization_method="q4_k_m")
```

**Lembrete:** arquivos criados no Colab somem quando a sessão acaba. Baixe a pasta `lora_triagem` se quiser guardá-la.

---
## Para pensar

1. O ajuste melhorou o **formato**, o **acerto**, ou os dois? Compare com o prompt v2 da Demo 2, que não exigiu treino nenhum.
2. Se a Secretaria recebesse 300 mil mensagens por mês, o que pesaria mais: o custo por token de uma API ou o custo de manter este modelo pequeno num servidor?
3. Se a regra de fronteira mudar (ex.: taxas passam a ir para outra fila), o que dá menos trabalho: mudar o prompt, re-indexar um RAG ou re-treinar o adaptador?